# ChronosAI — LLM-Powered Student Time-Management Prototype

**Module:** PUSL 3205 — Large Language Models and Generative AI
**Lab 02:** AI in Everyday Challenges (*The Busy Student*)
**Author:** Kumarage Isiwara Didulani

---

## Overview
This notebook implements a functional simulation of **ChronosAI**, an intelligent academic assistant designed to help university students manage overlapping lectures, coding labs, and project deadlines through four core pipelines:
1. **Natural-Language Task Extractor:** Converts unstructured student requests into structured JSON metadata via system prompt engineering.
2. **Dynamic Energy-Aware Scheduler:** Allocates tasks into cognitive-energy time blocks (Morning, Afternoon, Evening).
3. **Proactive Burnout & Overload Analyzer:** Flags weekly overcommitment before deadlines collide.
4. **Automated Morning Briefing Generator:** Synthesizes daily priorities and workload health into a concise action plan.

### Task 1: LLM System Prompt & Structured Task Extractor

**Objective:**
Construct a structured LLM system prompt and parsing pipeline that converts unstructured, natural-language student requests into clean JSON task metadata (`task_name`, `deadline`, `estimated_hours`, `priority`, `status`).

**Key Concepts:**
* **System Prompt Engineering:** Defines the model's persona (`ChronosAI`), extraction schema, and strict JSON output constraints.
* **Entity & Deadline Separation:** Isolates the core deliverable from temporal cues (e.g., *"before Friday afternoon"*) and infers task priority.

In [1]:
import json
import re

def parse_student_request(user_prompt: str) -> str:
    """
    Simulates an LLM Task Parsing Pipeline that extracts structured task metadata
    from unstructured natural-language student text.
    """
    system_prompt = (
        "You are ChronosAI, an intelligent academic task extraction engine. "
        "Extract task_name, deadline, estimated_hours, and priority (High/Medium/Low). "
        "Return valid JSON only."
    )

    # Extract core task name before deadline prepositions
    task_match = re.search(
        r'(?:finish|do|prepare|complete|submit)\s+(.*?)(?=\s+(?:due|on|by|before|for)\b|$)',
        user_prompt,
        re.IGNORECASE
    )
    task_name = task_match.group(1).strip() if task_match else "General Academic Task"

    # Extract deadline phrase if present
    deadline_match = re.search(
        r'\b(?:due|on|by|before)\s+(.*)$',
        user_prompt,
        re.IGNORECASE
    )
    deadline = deadline_match.group(1).strip().capitalize() if deadline_match else "Flexible / Next 48 Hours"

    # Infer priority from urgency and academic weight keywords
    prompt_lower = user_prompt.lower()
    if any(word in prompt_lower for word in ["urgent", "exam", "lab", "viva", "submission"]):
        priority = "High"
    elif any(word in prompt_lower for word in ["group", "meeting", "draft", "notes"]):
        priority = "Medium"
    else:
        priority = "Low"

    llm_output = {
        "system_prompt_used": system_prompt,
        "extracted_task": {
            "task_name": task_name,
            "deadline": deadline,
            "estimated_hours": 2.5,
            "priority": priority,
            "status": "Scheduled"
        }
    }

    return json.dumps(llm_output, indent=2)

# Test the improved task extraction pipeline
sample_request = "I need to complete my NLP Year 3 lab exercise before Friday afternoon"
print("--- Extracted LLM Task Output ---")
print(parse_student_request(sample_request))

--- Extracted LLM Task Output ---
{
  "system_prompt_used": "You are ChronosAI, an intelligent academic task extraction engine. Extract task_name, deadline, estimated_hours, and priority (High/Medium/Low). Return valid JSON only.",
  "extracted_task": {
    "task_name": "my NLP Year 3 lab exercise",
    "deadline": "Friday afternoon",
    "estimated_hours": 2.5,
    "priority": "High",
    "status": "Scheduled"
  }
}


### Task 2: Dynamic Energy-Aware Scheduler

**Question:**
> *How can an AI-powered assistant automatically distribute and schedule student tasks across optimal daily time blocks based on priority and cognitive energy levels?*

**Explanation:**
* **Energy-Based Allocation**: High-priority or complex technical tasks (like coding or machine learning labs) require high cognitive focus and are slotted into morning windows.
* **Workload Balancing**: Distributes medium and low-priority tasks into afternoon and evening slots to prevent cognitive burnout.

In [2]:
def schedule_tasks(tasks):
    """
    Dynamically schedules tasks into optimal time blocks based on priority and energy requirements.
    """
    schedule = {
        "Morning (High Focus ⚡)": [],
        "Afternoon (Collaboration 🤝)": [],
        "Evening (Review & Light Tasks 🌙)": []
    }

    for task in tasks:
        priority = task.get("priority", "Medium")
        if priority == "High":
            schedule["Morning (High Focus ⚡)"].append(task["task_name"])
        elif priority == "Medium":
            schedule["Afternoon (Collaboration 🤝)"].append(task["task_name"])
        else:
            schedule["Evening (Review & Light Tasks 🌙)"].append(task["task_name"])

    return schedule

# Sample student task pool for ChronosAI
sample_tasks = [
    {"task_name": "NLP Lab Transformer Implementation", "priority": "High"},
    {"task_name": "Group Project Git Branch Merge", "priority": "Medium"},
    {"task_name": "Review Lecture Slides (Week 2)", "priority": "Low"}
]

print("--- ChronosAI Optimized Schedule ---")
optimized_schedule = schedule_tasks(sample_tasks)
for time_block, items in optimized_schedule.items():
    print(f"\n{time_block}:")
    if items:
        for item in items:
            print(f"  • {item}")
    else:
        print("  • No tasks scheduled")

--- ChronosAI Optimized Schedule ---

Morning (High Focus ⚡):
  • NLP Lab Transformer Implementation

Afternoon (Collaboration 🤝):
  • Group Project Git Branch Merge

Evening (Review & Light Tasks 🌙):
  • Review Lecture Slides (Week 2)


### Task 3: Proactive Burnout & Overload Warning System

**Question:**
> *How do we analyze weekly student workload density to proactively detect overload and trigger automated burnout warnings before deadlines pile up[cite: 5]?*

**Explanation:**
* **Workload Aggregation**: Sums up estimated hours for all active assignments, labs, and group projects scheduled across the week.
* **Burnout Thresholding**: Compares total commitment hours against a sustainable weekly study limit (e.g., 35 hours) to flag high-risk overload states and suggest task deferral.

In [3]:
def check_workload_burnout(task_list, weekly_limit_hours=35.0):
    """
    Analyzes total weekly task hours and triggers burnout alerts if capacity is exceeded.
    """
    total_hours = sum(task.get("estimated_hours", 3.0) for task in task_list)

    print(f"Total Planned Study & Project Hours: {total_hours} hrs / week")
    print(f"Sustainable Capacity Limit: {weekly_limit_hours} hrs / week\n")

    if total_hours > weekly_limit_hours:
        return {
            "status": "⚠️ Burnout Risk Detected",
            "message": f"Your workload exceeds the safe limit by {total_hours - weekly_limit_hours} hours. ChronosAI recommends delegating or postponing low-priority tasks."
        }
    else:
        return {
            "status": "🟢 Balanced Workload",
            "message": "Your schedule is healthy and well-distributed. Keep up the great pace!"
        }

# Test with an overloaded student schedule
heavy_tasks = [
    {"task_name": "NLP Transformer Lab", "estimated_hours": 8.0},
    {"task_name": "Data Science Group Project API", "estimated_hours": 12.0},
    {"task_name": "Full-Stack Web App (Curate)", "estimated_hours": 10.0},
    {"task_name": "Exam Preparation (Data Structures)", "estimated_hours": 10.0}
]

evaluation = check_workload_burnout(heavy_tasks)
print(f"Workload Status: {evaluation['status']}")
print(f"AI Recommendation: {evaluation['message']}")

Total Planned Study & Project Hours: 40.0 hrs / week
Sustainable Capacity Limit: 35.0 hrs / week

Workload Status: ⚠️ Burnout Risk Detected
AI Recommendation: Your workload exceeds the safe limit by 5.0 hours. ChronosAI recommends delegating or postponing low-priority tasks.


### Task 4: Interactive Morning Briefing Generator

**Question:**
> *How do we synthesize scheduled tasks, priority metrics, and workload warnings into a concise, actionable morning briefing for the student?*

**Explanation:**
* **Context Synthesis**: Combines the outputs of task extraction, energy scheduling, and burnout checks into a single narrative summary.
* **Daily Actionability**: Highlights top-priority focus items so the student can start their day without decision fatigue.

In [4]:
def generate_morning_briefing(student_name, schedule_dict, burnout_status):
    """
    Generates an automated daily AI briefing for the student.
    """
    briefing = f"========================================\n"
    briefing += f"🤖 CHRONOS AI - DAILY MORNING BRIEFING\n"
    briefing += f"========================================\n"
    briefing += f"Good morning, {student_name}! Here is your optimized game plan for today:\n\n"

    for block, tasks in schedule_dict.items():
        briefing += f"📌 {block}:\n"
        if tasks:
            for task in tasks:
                briefing += f"   - {task}\n"
        else:
            briefing += f"   - No pending items (Free window)\n"

    briefing += f"\n📊 Workload Health Check: {burnout_status['status']}\n"
    briefing += f"💡 AI Insight: {burnout_status['message']}\n"
    briefing += f"========================================"

    return briefing

# Test the morning briefing generator
sample_briefing = generate_morning_briefing(
    student_name="Isiwara",
    schedule_dict=optimized_schedule,
    burnout_status=evaluation
)

print(sample_briefing)

🤖 CHRONOS AI - DAILY MORNING BRIEFING
Good morning, Isiwara! Here is your optimized game plan for today:

📌 Morning (High Focus ⚡):
   - NLP Lab Transformer Implementation
📌 Afternoon (Collaboration 🤝):
   - Group Project Git Branch Merge
📌 Evening (Review & Light Tasks 🌙):
   - Review Lecture Slides (Week 2)

📊 Workload Health Check: ⚠️ Burnout Risk Detected
💡 AI Insight: Your workload exceeds the safe limit by 5.0 hours. ChronosAI recommends delegating or postponing low-priority tasks.
